In [11]:
import torch
import torch.nn as nn
import torch.nn.functional as F

with open("alice.txt", "r", encoding="utf-8") as f:
    texte = f.read()

print("Nombre de caractères :", len(texte))
print("Aperçu :\n", texte[:300])


Nombre de caractères : 593
Aperçu :
 Alice was beginning to get very tired of sitting by her sister on the bank, and of having nothing to do: once or twice she had peeped into the book her sister was reading, but it had no pictures or conversations in it, “and what is the use of a book,” thought Alice “without pictures or conversations


In [12]:
caracteres_uniques = sorted(set(texte))
char_to_id = {c: i for i, c in enumerate(caracteres_uniques)}
id_to_char = {i: c for c, i in char_to_id.items()}

print("Taille du vocabulaire :", len(char_to_id))

phrase = "Alice was beginning to get very tired"

tokens = [char_to_id.get(c, 0) for c in phrase]

print("Phrase :", phrase)
print("Tokens :", tokens)

texte_decode = ''.join([id_to_char.get(t, '?') for t in tokens])
print("Texte décodé :", texte_decode)

Taille du vocabulaire : 37
Phrase : Alice was beginning to get very tired
Tokens : [9, 23, 21, 15, 17, 1, 33, 13, 29, 1, 14, 17, 19, 21, 25, 25, 21, 25, 19, 1, 30, 26, 1, 19, 17, 30, 1, 32, 17, 28, 34, 1, 30, 21, 28, 17, 16]
Texte décodé : Alice was beginning to get very tired


In [13]:
class SimpleLLM(nn.Module):
    def __init__(self, vocab_size, embed_dim=64, hidden_dim=128):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.fc = nn.Linear(embed_dim, hidden_dim)
        self.out = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x):
        h = self.embed(x)
        h = torch.relu(self.fc(h))
        return self.out(h)

vocab_size = len(char_to_id)
model = SimpleLLM(vocab_size)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)


In [14]:
seq_len = 64
batch_size = 64

for epoch in range(200):
    idx = torch.randint(0, len(data) - seq_len - 1, (batch_size,))
    x = torch.stack([data[i:i+seq_len] for i in idx])
    y = torch.stack([data[i+1:i+seq_len+1] for i in idx])
    
    logits = model(x)
    loss = F.cross_entropy(logits.view(-1, vocab_size), y.view(-1))
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    if epoch % 20 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")


Epoch 0, Loss: 3.6007
Epoch 20, Loss: 2.7158
Epoch 40, Loss: 2.3118
Epoch 60, Loss: 2.1579
Epoch 80, Loss: 2.0956
Epoch 100, Loss: 1.9839
Epoch 120, Loss: 1.9703
Epoch 140, Loss: 1.9608
Epoch 160, Loss: 1.9223
Epoch 180, Loss: 1.9049


In [15]:
torch.save({
    'model': model.state_dict(),
    'char_to_id': char_to_id
}, 'model_alice.pt')


In [16]:
def generate(model, prompt, max_tokens=200, temperature=1.0):
    model.eval()
    tokens = [char_to_id.get(c, 0) for c in prompt]

    with torch.no_grad():
        for _ in range(max_tokens):
            x = torch.tensor([tokens[-64:]], dtype=torch.long)
            logits = model(x)[0, -1, :]
            probs = F.softmax(logits / temperature, dim=-1)
            next_token = torch.multinomial(probs, 1).item()
            tokens.append(next_token)

    return ''.join([id_to_char.get(t, '?') for t in tokens])


In [18]:
print("Température 0.3\n")
print(generate(model, "Alice was ", max_tokens=200, temperature=0.3))

print("\nTempérature 0.8\n")
print(generate(model, "Alice was ", max_tokens=200, temperature=0.8))

print("\nTempérature 1.3\n")
print(generate(model, "Alice was ", max_tokens=200, temperature=1.3))


Température 0.3

Alice was be the othe the ould be the the ing s the s sithe the he whe ping ong or s whe ad pinve of our ping of he of ting our wa or ing ton the the ing he s ong s ang the the per o ong wher t ang the the he t

Température 0.8

Alice was athe d), bo o ble o t our s f asithe hor theldin vether than iche suplde trid s orepisuge g, or wour tug dat at thowoupe ing oulller aing inshepito f oong sthee ma con f d, ouper p asinge cotronder co

Température 1.3

Alice was g, o bl pid cthait, asaler watherir sit wheeperedadas on ghaindele git ticn ply ohen ceghe d), 


isad ce oursheron ur), oudathougheering stn a ctr howadincokin ie p we beche h, ower d pert ok A bad a
